# 02 — Satellite azimuthal-anisotropy fits: TNG100 ($>10^8$) & TNG50 ($>10^7$)

Reads the all-central catalogs from `01_generate_allcentral_catalogs.ipynb` (in `../data2`) and
fits the **azimuthal anisotropy** of the satellite distribution,
$p(\theta)\propto 1 + A\cos 2\theta$ on $\theta\in[0,90]$ ($0^\circ$ = host major axis). $A>0$ is a
major-axis excess; $A=0$ is isotropic. The amplitude $A$ is sampled with `emcee` (unbinned
likelihood, same as `notebooks/16`).

**What is plotted (two panels):**
* **Left — TNG100**, satellites $M_{*,\rm sat} > 10^8\,M_\odot$, at **z = 0** and **z = 0.05**.
* **Right — TNG50**, satellites $M_{*,\rm sat} > 10^7\,M_\odot$, at **z = 0** and **z = 0.05**.

**Radius cut toggle.** `RADIUS_CUT = False` by default → **no $R_{200c}$ cut** (all radii). Set it
to `True` to keep only satellites within `R200C_FACTOR` $\times R_{200c}$ (3D). Because notebook 01
writes satellites at all radii and stores `d_r200_3d`, this is a pure post-selection — no
regeneration needed.

> "z = 0.5" is interpreted as **z = 0.05** (snap 98), consistent with notebook 01. Missing catalogs
> are skipped with a warning, so this runs on whatever `../data2` currently holds.

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import emcee
import matplotlib as mpl
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
%matplotlib inline
mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "mathtext.fontset": "cm",
    "axes.unicode_minus": False,
})

# --- configuration -----------------------------------------------------------
DATA_ROOT = "../data2"
CAT_TAG   = "logM7.00"                     # the mass floor notebook 01 wrote (satellites > 1e7)
CAT_NAME  = f"tng_satellites_allcentrals_{CAT_TAG}.csv"

# radius-cut toggle: False = no R200c cut (all radii); True = keep d_r200_3d < R200C_FACTOR
RADIUS_CUT   = False
R200C_FACTOR = 1.0

# what each panel shows: (sim, satellite log10 M* cut, panel color)
PANELS = [
    ("tng100", 8.0, "#1f77b4"),           # TNG100: M*_sat > 1e8
    ("tng50",  7.0, "#d62728"),           # TNG50:  M*_sat > 1e7
]
# redshifts overlaid within each panel: (subdir, label, linestyle, marker)
REDS = [("z0", "z = 0", "-", "o"), ("z0p05", "z = 0.05", "--", "s")]

# 18 angle bins over [0, 90]
N_BINS      = 18
ANGLE_EDGES = np.linspace(0, 90, N_BINS + 1)

print(f"radius cut: {'d_r200_3d < %.2f R200c' % R200C_FACTOR if RADIUS_CUT else 'NONE (all radii)'}")

## Helpers — $P(\theta)$, the model curve, and the unbinned MCMC $A$ fit

Same anisotropy machinery as `notebooks/16`.

In [ ]:
def norm_hist(theta):
    c, _ = np.histogram(theta, bins=ANGLE_EDGES)
    return c / c.sum() / (90.0 / N_BINS)

def model_curve(A):
    x = np.linspace(0, 90, 200)
    return x, (1.0 / 90.0) * (1.0 + A * np.cos(2 * np.radians(x)))

def fit_anisotropy(theta_deg, n_walkers=16, n_steps=4000, burn=1000, seed=0):
    '''MCMC of p(theta) ~ 1 + A cos(2 theta). Returns dict with A median/std/percentiles.'''
    th = np.radians(np.asarray(theta_deg, dtype=float))
    c2 = np.cos(2 * th); n = len(th)
    if n < 5:
        return dict(n=n, A=np.nan, Aerr=np.nan, lo=np.nan, hi=np.nan)

    def log_prob(p):
        A = p[0]
        if not (-0.999 < A < 0.999):
            return -np.inf
        v = 1.0 + A * c2
        if np.any(v <= 0):
            return -np.inf
        return np.sum(np.log(v))

    rng = np.random.default_rng(seed)
    p0 = rng.uniform(-0.1, 0.1, size=(n_walkers, 1))
    sampler = emcee.EnsembleSampler(n_walkers, 1, log_prob)
    sampler.run_mcmc(p0, n_steps, progress=False)
    chain = sampler.get_chain(discard=burn, flat=True)[:, 0]
    lo, med, hi = np.percentile(chain, [16, 50, 84])
    return dict(n=n, A=med, Aerr=chain.std(), lo=lo, hi=hi)

## Load the catalogs and apply the satellite mass cut (and optional radius cut)

For each panel we read the single all-central catalog per redshift, keep `mstar_phys > cut`, and —
if `RADIUS_CUT` — also `d_r200_3d < R200C_FACTOR`. Missing files are skipped with a warning.

In [ ]:
def load_theta(sim, zsub, logcut):
    path = os.path.join(DATA_ROOT, sim, zsub, CAT_NAME)
    if not os.path.exists(path):
        print(f"[skip] missing {path}")
        return None
    df = pd.read_csv(path)
    df = df[df["mstar_phys"] > logcut]
    if RADIUS_CUT:
        df = df[df["d_r200_3d"] < R200C_FACTOR]
    return df["alpha"].to_numpy()

data, fits = {}, {}
for sim, logcut, _ in PANELS:
    for zsub, zlabel, _, _ in REDS:
        key = (sim, zsub)
        th = load_theta(sim, zsub, logcut)
        if th is None or len(th) == 0:
            continue
        data[key] = th
        fits[key] = fit_anisotropy(th, seed=1)

print(f"{'dataset':<18s} {'N':>6s}  {'A':>16s}   |A|/sig")
for sim, logcut, _ in PANELS:
    for zsub, zlabel, _, _ in REDS:
        key = (sim, zsub)
        if key not in fits:
            continue
        f = fits[key]
        sig = abs(f["A"] / f["Aerr"]) if np.isfinite(f["A"]) else np.nan
        print(f"{sim+' '+zlabel:<18s} {f['n']:>6d}  {f['A']:+.3f} +/- {f['Aerr']:.3f}   {sig:.2f}")

## Figure — TNG100 ($>10^8$) vs TNG50 ($>10^7$), each at z=0 and z=0.05

Stepped histogram = $P(\theta)$; smooth curve = posterior-median $\tfrac{1}{90}(1+A\cos2\theta)$;
dotted line = isotropic level. Solid = z=0, dashed = z=0.05.

In [ ]:
SATLABEL = {8.0: r"$M_* > 10^8\,M_\odot$", 7.0: r"$M_* > 10^7\,M_\odot$"}

fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
for ax, (sim, logcut, color) in zip(axes, PANELS):
    for zsub, zlabel, ls, marker in REDS:
        key = (sim, zsub)
        if key not in fits:
            continue
        f = fits[key]
        h = norm_hist(data[key])
        ax.step(ANGLE_EDGES, np.r_[h, h[-1]], where="post", color=color, ls=ls, lw=1.5,
                alpha=0.6, label=f"{zlabel}  (N={f['n']}, A={f['A']:+.2f})")
        if np.isfinite(f["A"]):
            x, y = model_curve(f["A"]); ax.plot(x, y, color=color, ls=ls, lw=2.5)
    ax.axhline(1 / 90, color="k", lw=1, ls=":")
    ax.set_xlim(0, 90); ax.set_ylim(0, 0.025)
    ax.set_xlabel(r"$\theta$ [deg]")
    ax.set_title(f"{sim.upper()}   ({SATLABEL[logcut]})")
    ax.legend(fontsize=9, fancybox=False, edgecolor="k")
    ax.tick_params(which="both", direction="in", top=True, right=True)
axes[0].set_ylabel(r"$P(\theta)$")
_rc = f"within {R200C_FACTOR:g} R200c" if RADIUS_CUT else "no radius cut"
fig.suptitle(f"Satellite azimuthal anisotropy  (all centrals, {_rc})", y=1.02, fontsize=14)
plt.subplots_adjust(wspace=0.08)
plt.show()